In [1]:
import numpy as np
import glob
import open3d as o3d
import cv2
import matplotlib.pyplot as plt

In [ ]:
path_117322070663 = "./mask/108322073284_*.ply"
path_108322073284 = "./mask/108322073284_*.ply"

In [ ]:
camera_params = {
    "117322070663": {
        "K": np.array([[591.2588416325026, 0.0, 303.96766080041255], [0.0, 582.3094248920196, 253.6549343618132], [0.0, 0.0, 1.0]]),
        "D": np.array([0.0857423535202526, 0.15899390827577137, 0.0027644426081692413, -0.014441335296417794, -1.3507539704153515])
    },
    "108322073284": {
        "K": np.array([[581.4988972720037, 0.0, 339.4397798655603], [0.0, 582.8600048067065, 282.73657769938495], [0.0, 0.0, 1.0]]),
        "D": np.array([-0.24057957220615858, 4.2767106826695604, 0.024711451849462666, 0.013758143071840522, -20.808592661502967])
    }
}

R = np.array([
    [-0.0326154191220418, 0.6047670517721788, -0.7957342819850609],
    [-0.581149899271677, 0.6362593173876714, 0.5073843470327937],
    [0.8131426867481583, 0.47898945095707646, 0.3307084469133276]
])

t = np.array([0.26853285220651585, -0.174979423376861, 0.1982699183045584]).reshape(3, 1)

In [ ]:
Rt = np.array([
    [-0.0326154191220418, 0.6047670517721788, -0.7957342819850609, 0.26853285220651585],
    [-0.581149899271677, 0.6362593173876714, 0.5073843470327937, -0.174979423376861],
    [0.8131426867481583, 0.47898945095707646, 0.3307084469133276, 0.1982699183045584],
    [0, 0, 0, 1]
], dtype=float)

In [ ]:
def camera_to_image(K, ptc, name="Depth Map"):
    X = ptc[:, 0]
    Y = ptc[:, 1]
    Z = ptc[:, 2]

    Z_abs = np.abs(Z)

    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    
    u = ((fx * X) / Z_abs) + cx
    v = ((fy * Y) / Z_abs) + cy

    depth_map = np.zeros((480, 640))
    u_int = np.round(u).astype(int)
    v_int = np.round(v).astype(int)
    
    mask = (u_int >= 0) & (u_int < 640) & (v_int >= 0) & (v_int < 480)
    
    depth_map[v_int[mask], u_int[mask]] = Z_abs[mask]
    
    depth_map[depth_map == 0] = np.nan

    plt.figure(figsize=(8, 6))
    img = plt.imshow(depth_map, cmap='jet')
    plt.colorbar(img, label='Distanza (m)')
    plt.title(name)
    plt.show()

    return u, v

In [ ]:
camera_117322070663 = []

for element in glob.glob(path_117322070663):
    pcd = o3d.io.read_point_cloud(element)
    points = np.asarray(pcd.points)

    camera_117322070663.append(points)

In [ ]:
camera_108322073284 = []

for element in glob.glob(path_108322073284):
    pcd = o3d.io.read_point_cloud(element)
    points = np.asarray(pcd.points)

    camera_108322073284.append(points)

In [ ]:
for element in camera_117322070663:
    u, v = camera_to_image(camera_params["117322070663"]["K"], element)

In [ ]:
for element in camera_108322073284:
    u, v = camera_to_image(camera_params["108322073284"]["K"], element)